# Module 4.5 - User Personalisation

**Reference:** [`05-user-personalisation.md`](./05-user-personalisation.md)

## What you'll do in this notebook

- Use the LLM to extract durable user facts (technical level, preferences, constraints, goals) from conversation history.
- Persist a user profile to disk as JSON.
- Inject the profile into the chatbot's system prompt and compare personalised vs generic answers.

## Setup

In [ ]:
import os
import json
from datetime import datetime
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set."

client = OpenAI()
MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
PROFILE_DIR = Path("./profiles_m4_5")
PROFILE_DIR.mkdir(exist_ok=True)
print(f"OK: profiles will be saved under {PROFILE_DIR}")

## Exercise 1 - Fact extraction

Ask the LLM to read a short conversation and return a JSON-shaped list of durable facts about the user. Ask for strict JSON so you can parse the reply.

In [ ]:
EXTRACT_SYS = (
    "You extract a concise user profile from chat conversations. "
    "Return STRICT JSON of shape {\"facts\": [\"short factual statement about the user\", ...]}. "
    "Facts must be durable (technical level, preferences, constraints, goals, stack, OS). "
    "Do NOT include transient state (what they're doing right now, the current error)."
)

def extract_facts(messages: list[dict]) -> list[str]:
    transcript = "\n".join(f"{m['role']}: {m['content']}" for m in messages)
    # TODO:
    # 1. Call client.chat.completions.create with:
    #    model=MODEL, temperature=0,
    #    response_format={"type": "json_object"},
    #    messages=[{system=EXTRACT_SYS}, {user=transcript}]
    # 2. json.loads the reply and return payload.get("facts", []).
    raise NotImplementedError

sample = [
    {"role": "user",      "content": "I'm a data scientist at a healthcare startup."},
    {"role": "assistant", "content": "Great, happy to help. What are you working on?"},
    {"role": "user",      "content": "I prefer pandas over raw SQL. I'm on a MacBook. Any chance we can stick to Python examples?"},
    {"role": "assistant", "content": "Absolutely, I'll default to Python."},
    {"role": "user",      "content": "I'm still learning async, so please keep examples synchronous for now."},
]
for f in extract_facts(sample):
    print("-", f)

## Exercise 2 - Persist the profile to disk

One JSON file per user. Accumulate facts; deduplicate conservatively (exact string match).

In [ ]:
class UserProfile:
    def __init__(self, user_id: str):
        self.user_id = user_id
        self.path = PROFILE_DIR / f"user_{user_id}.json"
        self.data = self._load()

    def _load(self) -> dict:
        if self.path.exists():
            return json.loads(self.path.read_text())
        return {"user_id": self.user_id, "facts": [], "updated_at": None}

    def _save(self) -> None:
        self.data["updated_at"] = datetime.now().isoformat()
        self.path.write_text(json.dumps(self.data, indent=2))

    def add_facts(self, new_facts: list[str]) -> None:
        # TODO: lowercase-compare each new_fact against self.data["facts"];
        # append only those not already present; then self._save().
        raise NotImplementedError

    def summary(self) -> str:
        if not self.data["facts"]:
            return ""
        return "User profile:\n" + "\n".join(f"- {fact}" for fact in self.data["facts"])

profile = UserProfile("alex")
profile.add_facts(extract_facts(sample))
print(profile.summary())

## Exercise 3 - Personalised chatbot

Inject the profile into the system prompt and compare replies to the *same* question with and without it.

In [ ]:
def ask(system_prompt: str, user_message: str) -> str:
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": user_message}],
        temperature=0.4,
    )
    return resp.choices[0].message.content

QUESTION = "Which database should I use?"

base_prompt = "You are a helpful programming assistant. Keep replies to 4-6 lines."
personalised_prompt = base_prompt + "\n\n" + profile.summary() + "\n\nTailor answers to the user's profile."

print("-- generic --")
print(ask(base_prompt, QUESTION))
print("\n-- personalised --")
print(ask(personalised_prompt, QUESTION))

**Reflection:**

- Does the personalised answer cite the user's stack (Python, pandas, MacBook)?
- How would you handle a contradiction - e.g. the user later says 'actually I switched to Rust'? Sketch a 'consolidate profile' step that asks the LLM to merge the list of facts, resolving conflicts.

## Capstone checkpoint 4

Your capstone chatbot should now handle long conversations sensibly:

1. Pick **two** memory strategies from this module (the capstone requires auto-summarisation in section 4.2 plus one more of your choice).
2. Plug them into the Streamlit app from Module 8. For summarisation, trigger when history crosses a configurable turn threshold.
3. Run a 40-turn smoke test: confirm the context window never overflows and a fact from turn 1 is still reachable at turn 40.
4. Record the experiment results (turns, peak tokens, whether the recall test passed) - these go in the capstone results table.

## What's next

You now have every piece the capstone needs: LLM calls with retries (M1), RAG over ChromaDB (M2), optional advanced retrieval techniques (M3), and memory + personalisation (M4). The capstone brief at `module_8/01-capstone-project-brief.md` walks you through assembling them into a Streamlit-based domain chatbot.